<a href="https://colab.research.google.com/github/SotaYoshida/Lecture_DataScience/blob/master/Python_chapter_WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web操作・スクレイピング

Webから情報を抽出・整形・解析したり、  
ブラウザ上での特定の操作を自動化する、といったことも  
Pythonでは比較的容易に実行することができる。

Web上にある情報にアクセスしたりする方法は色々あるが、ここでは大まかに以下の２つに分類する:

1. プログラムでWebページにアクセスして、ソースコード(HTML)を読み、
そこから情報を抽出する方法
2. ブラウザ(Google Chromeとか)をプログラムに操作させて特定の作業を実行する方法

この章では、とくに1.のWebから情報を抽出すること(スクレイピング)に絞ってそのエッセンスを紹介する。  

Webから情報を抽出したりする際、共通して言える注意点を述べておく:
* **対象とするページの利用規約を必ず確認する**  
規約でスクレイピングを禁止しているページがある (例: Amazon, Twitter, Instagram, facebook, 金融系などなど)  
    禁止している場合でも、APIが提供されている場合があります  
    ※APIはApplication Programming Interfaceの略です。  
    今の場合、大雑把にはデータ提供用の窓口とでも思ってください

* **サーバーに負荷をかけない**  
    規約で特にスクレイピングを禁止していない場合でも、過度なアクセスをしてはいけません。  
    (どこかの大学の学習支援システムみたいに落ちてしまったら大変です)   
    過度なアクセスは、悪意のあるDos攻撃とみなされてアクセスを制限されたり、  
    最悪の場合、偽計業務妨害罪などの罪に問われる可能性があります。






## 国立感染症研究所からのインフルエンザの感染報告者数の取得


スクレイピングを可能とするライブラリは多数存在する。  
代表的なものは```requests```,```urllib```,```BeautifulSoup4```などなど。

以下では、```requests```を使います。  
JavaScriptの実行などがないページならこれでだいたい十分かと思います。

ややadhocな実装ですが、
[国立感染症研究所のインフルエンザ流行レベルマップ](https://nesid4g.mhlw.go.jp/Hasseidoko/Levelmap/flu/)から  
各週のインフルエンザ患者数を抽出する以下のコードを実行してみましょう。

In [ ]:
import requests
import pandas as pd
import time
import re #正規表現というものを使うためのモジュール

def exnum(a): #文字列から数値を抽出する関数
    pattern = r'(\d{1,3}(,\d{3})*)'
    return re.findall(pattern,a)



#スクレイピングする区間の指定
years = range(2010,2020)
#years = range(2017,2020)
sw = 35; ew = 28

weeks = [j for j in range(sw,53)] + [i for i in range(1,ew)]
hit =0 
sumV=[]
for s_year in years:
    for week in weeks:
        if week < 10:
            c_week = "0"+str(week)
        else:
            c_week = str(week)
        if s_year < 2007:
            tw= str(c_week)
        else:
            tw=""
        if week <=ew :
            c_year = str(s_year+1)
        else:
            c_year = str(s_year)
        time.sleep(0.5) ###サイトに負荷をかけないように間隔をあける
        url = 'https://nesid4g.mhlw.go.jp/Hasseidoko/Levelmap/flu/'+str(s_year)+'_'+str(s_year+1)+'/'+c_year+'_'+c_week+'/jmap'+tw+'.html'
        ctext = str(c_year)+"年"+c_week+"週目: "
        try :
            response = requests.get(url)
            response.encoding = response.apparent_encoding
            lines = response.text
            tmp = lines.split("患者報告数")
            num = exnum(tmp[1][:20])[0]            
            hit += 1
            try :
                if type(num) is tuple:
                    num = num[0]
                ctmp = str( int(num.replace(",","")))
                ctext += ctmp
            except:
                print("error in try")
        except:
            ctext += ""
            num = ""
        sumV += [ [ c_year, c_week, num] ]
        print(ctext)
print("hit=",hit)        
df = pd.DataFrame(sumV)





2010年35週目: 
2010年36週目: 202
